# Impulse — Reporting Pipeline Demo

The **Impulse Framework** is a Python library that enables
automotive and industrial engineers to process, aggregate,
and analyze petabytes of time-series measurement data on
Databricks — without requiring Apache Spark expertise.

It provides **TSAL** (Time Series Analytics Language),
a Pythonic expression language for defining signals,
events, and aggregations.

**What this notebook builds:**
A complete reporting pipeline — RPM histograms,
RPM-vs-speed heatmaps, per-distance-bin statistics, and
channel values sampled at every 10 km milestone — across
3 test drives, persisted as a Gold-layer star schema and
visualized inline with matplotlib.

**Requirements:**
- Databricks workspace with **Unity Catalog** access
- **Serverless** compute (or DBR 14+)

## Architecture

![Impulse architecture](../docs/img/architecture.png)

Impulse sits between a governed silver layer and a gold-layer star schema in Unity Catalog and provides three components:

- **TSAL (Time Series Analytics Language)** — a declarative Python DSL for expressing signals, events, and aggregations in natural Python, without requiring Spark expertise.
- **Query Engine** — pluggable and distributed; compiles TSAL expressions into Spark execution plans and adapts to any silver-layer layout via interchangeable solvers.
- **Aggregations** — domain-aware physical aggregations, including duration- and distance-weighted 1D/2D histograms and event-scoped statistics.

# 1. Setup

In [0]:
%pip install pydantic>=2.0 scipy -q
dbutils.library.restartPython()

### Configure target location

Fill in **Catalog**, **Schema**, and **Table Prefix**
in the widgets above, then run the next cells.

In [0]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text(
    "table_prefix", "", "Table Prefix"
)
dbutils.widgets.dropdown("drop_created_tables", "false", ["true", "false"], "Drop Created Tables")

In [0]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import sys, os

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_PREFIX = dbutils.widgets.get("table_prefix")

if not CATALOG or not SCHEMA:
    raise ValueError(
        "Please set Catalog and Schema "
        "widgets above before running."
    )

nb_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().notebookPath().get()
)
DEMOS_DIR = (
    "/Workspace"
    + "/".join(nb_path.split("/")[:-1])
)
REPO_ROOT = (
    "/Workspace"
    + "/".join(nb_path.split("/")[:-2])
)
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

pfx = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}"
print(f"Target: {pfx}_*")

### Load demo data into Silver layer

Impulse reads from a **Silver layer**:
- **Container** = one measurement recording
  (e.g., one test drive)
- **Channel** = one sensor signal within a container
  (e.g., Engine RPM), stored as raw
  `(timestamp, value)` samples — the framework
  automatically converts these to intervals on the fly

In [0]:
import pandas as pd

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{SCHEMA}"
)

csv_dir = os.path.join(DEMOS_DIR, "data", "reporting")

SILVER = [
    "container_metrics", "container_tags",
    "channel_metrics", "channel_tags",
    "channels", "poi_channels",
]
for t in SILVER:
    pdf = pd.read_csv(f"{csv_dir}/{t}.csv")
    spark.createDataFrame(pdf).write.mode(
        "overwrite"
    ).saveAsTable(f"{pfx}_{t}")

print(f"Loaded {len(SILVER)} tables")

In [0]:
display(spark.read.table(f"{pfx}_container_metrics"))

In [0]:
display(spark.read.table(f"{pfx}_channels").limit(10))

## Pipeline Overview

The diagram below shows the end-to-end flow:
**Silver Layer** inputs and **user-defined** channels
and aggregations feed into Impulse Framework,
which produces a **Gold Layer** star schema.

![MDA Pipeline Overview](images/mda_pipeline_overview.png)

# 2. Initialize the Report

The `Report` orchestrator takes a config specifying:
- **`source`** — Silver layer tables
- **`unity_sink`** — Gold layer output
- **`query_engine.solver`** — `DefaultSolver` for
  parallel per-container execution
- **`query_engine.data_type`** — `RAW` for raw
  timestamp data (auto-converted to intervals)
- **`measurement_dimensions`** — container metadata
  to carry into Gold layer

In [0]:
from databricks.sdk import WorkspaceClient
from impulse_reporting.core.report import Report
from impulse_reporting.core.page import Page
from impulse_reporting.aggregations.histogram import HistogramDuration
from impulse_reporting.aggregations.histogram2d import Histogram2DDuration
from impulse_reporting.aggregations.stats_aggregator import StatsAggregator
from impulse_reporting.aggregations.point_value_aggregator import PointValueAggregator
from impulse_reporting.events.basic_event import BasicEvent
from impulse_reporting.events.container_event import ContainerEvent
from impulse_reporting.events.points_in_time_event import PointsInTimeEvent
import pyspark.sql.functions as F

ws = WorkspaceClient()

config = {
    "source": {
        "container_metrics_table": f"{pfx}_container_metrics",
        "channel_metrics_table": f"{pfx}_channel_metrics",
        "channels_uri": f"{pfx}_channels",
        "poi_channels_uri": f"{pfx}_poi_channels",
        "container_tags_table": f"{pfx}_container_tags",
        "channel_tags_table": f"{pfx}_channel_tags",
    },
    "unity_sink": {
        "catalog": CATALOG,
        "schema": SCHEMA,
        "table_prefix": TABLE_PREFIX,
    },
    "query_engine": {
        "solver": "DefaultSolver",
        "data_type": "RAW",
    },
    "measurement_dimensions": [
        "container_id", "vehicle_key",
        "start_ts", "stop_ts",
    ],
    "calculated_channels": {
        "emit_channel_metrics": True,
        "attribute_columns": [],
        "kpis": ["duration", "min", "max", "mean"],
    },
}

report = Report(
    name="cuj1_demo", spark=spark, config=config,
    workspace_client=ws
)
db = report.get_db()
print("Report initialized")

# 3. Select Physical Channels

Channels are selected by **metadata tags** —
no column names, no SQL, no joins.
These are **lazy expressions**: no data is read yet.

In [0]:
eng_rpm = db.query.channel(
    channel_name="Engine RPM",
    brand="Seat", model="Leon",
)
veh_spd = db.query.channel(
    channel_name="Vehicle Speed Sensor",
    brand="Seat", model="Leon",
)
amb_air_temp = db.query.channel(
    channel_name="Ambient Air Temperature",
    brand="Seat", model="Leon",
)
intake_air_temp = db.query.channel(
    channel_name="Intake Air Temperature",
    brand="Seat", model="Leon",
)

## 3a. Select POI Channels (Diagnostic Trouble Codes)

Not every channel is a continuous signal. A **Points-in-Time (POI)** channel is
an *event stream*: each value exists **only at its timestamp**, with no validity
in between. The textbook example is **DTCs** (Diagnostic Trouble Codes) — the
fault codes an ECU emits at the instant it detects a problem (`P0301` = cylinder-1
misfire, …).

POI channels are selected with **`poi_channel(...)`** instead of `channel(...)`.
Identification is identical (same metadata tags); only the semantics differ —
a POI channel solves to a `PointsInTimeSeries`, not a `SampleSeries`.

| | `channel(...)` | `poi_channel(...)` |
|---|---|---|
| Shape | `[tstart, tend)` intervals | `(tᵢ, vᵢ)` points |
| Valid between points? | yes (interpolated) | **no** |
| Backed by | `SampleSeries` | `PointsInTimeSeries` |

In [ ]:
# String POI channel: the DTC code emitted at each fault instant.
# dtype="string" -> equality is the natural operation ("when did P0301 occur?"),
# never arithmetic or ordering on a code.
dtc = db.query.poi_channel(
    channel_name="DTC",
    dtype="string",
    brand="Seat", model="Leon",
)

# Numeric POI channel: a running fault-occurrence counter.
dtc_count = db.query.poi_channel(
    channel_name="DTC_count",
    brand="Seat", model="Leon",
)

# 4. Define Virtual Signals & Events

**TSAL** uses Python operators to build lazy
expression trees — no Spark knowledge needed.

**Virtual signals** derive from physical channels.
**Events** are time windows where a condition holds.

In [0]:
avg_temp = (amb_air_temp + intake_air_temp) / 2
distance_km = (
    veh_spd.resample(1e6).cumtrapz() / 3600 / 1e6
)

rpm_band = (eng_rpm > 2000) & (eng_rpm < 5000)
every_10km = (
    (distance_km % 10)
    .intervals_between_falling_edges()
)

# Instant the trip odometer crosses each additional
# 10 km — a set of points in time, not an interval.
distance_milestones = (distance_km % 10).falling_edges()

# 5. Register Events

- **BasicEvent** — from a TSAL boolean expression
- **ContainerEvent** — spans the entire recording
- **PointsInTimeEvent** — a set of instants (e.g. each 10 km milestone)
- **PointsInTimeEvent** — a set of instants (e.g. each 10 km milestone, or each **P0301 misfire** from the DTC channel)

In [0]:
rpm_event = BasicEvent(
    name="rpm_event",
    expr=rpm_band,
    desc="Engine RPM between 2000 and 5000",
    required_channels=["Engine RPM"],
)
report.add_event(rpm_event)

container_event = ContainerEvent(
    name="container_event",
    desc="Full measurement recording",
)
report.add_event(container_event)

distance_event = BasicEvent(
    name="distance_event",
    expr=every_10km,
    desc="Every 10 km driven",
)
report.add_event(distance_event)

milestone_event = PointsInTimeEvent(
    name="distance_milestones",
    expr=distance_milestones,
    desc="Each 10 km driven (instant)",
)
report.add_event(milestone_event)

# POI freeze-frame: the instants a P0301 misfire was logged.
# `dtc == "P0301"` is a PointsInTime — the set of timestamps where the
# string code equals P0301 — exactly what a PointsInTimeEvent wants.
p0301_event = PointsInTimeEvent(
    name="p0301_misfires",
    expr=(dtc == "P0301"),
    desc="Each instant a P0301 misfire code was set",
)
report.add_event(p0301_event)

# 6. Define Aggregations

- **Histogram** — 1D duration-weighted distribution
- **Histogram2D** — 2D heatmap of two signals
- **StatisticsAggregator** — min, median, mean, max per event
- **PointValueAggregator** — channel value sampled at each instant of a points-in-time event

In [0]:
page = Page(page_number=1)
report.add_page(page)

sigs = [eng_rpm, veh_spd, amb_air_temp,
        intake_air_temp, avg_temp]
names = ["Engine RPM", "Vehicle Speed",
         "Ambient Air Temp", "Intake Air Temp",
         "Avg Temp"]
aggs = ["min", "median", "mean", "max"]

page.add_aggregation(HistogramDuration(
    name="rpm_histogram", base_expr=eng_rpm,
    bins=[float(i) for i in range(0, 5000, 250)],
    event=rpm_event,
    desc="RPM distribution within RPM events",
    channel_name="Engine RPM",
    bins_unit="RPM", values_unit="s",
))
page.add_aggregation(HistogramDuration(
    name="speed_histogram", base_expr=veh_spd,
    bins=[float(i) for i in range(0, 200, 10)],
    event=rpm_event,
    desc="Speed distribution within RPM events",
    channel_name="Vehicle Speed",
    bins_unit="km/h", values_unit="s",
))
page.add_aggregation(Histogram2DDuration(
    name="rpm_speed_heatmap",
    x_expr=eng_rpm, y_expr=veh_spd,
    x_bins=[float(i) for i in range(2000, 5000, 250)],
    y_bins=[float(i) for i in range(0, 200, 10)],
    event=rpm_event, desc="RPM vs Speed heatmap",
    x_channel_name="Engine RPM",
    y_channel_name="Vehicle Speed",
    x_bins_unit="RPM", y_bins_unit="km/h",
    values_unit="s",
))

page.add_aggregation(StatsAggregator(
    name="rpm_event_stats", input_expressions=sigs,
    channel_names=names, statistics=aggs,
    event=rpm_event,
    desc="Statistics within RPM events",
))
page.add_aggregation(StatsAggregator(
    name="container_stats", input_expressions=sigs,
    channel_names=names, statistics=aggs,
    event=container_event,
    desc="Statistics for full measurement",
))
page.add_aggregation(StatsAggregator(
    name="distance_stats",
    input_expressions=sigs + [distance_km],
    channel_names=names + ["Distance"],
    statistics=aggs,
    event=distance_event,
    desc="Statistics per 10 km distance bin",
))

# Sample Vehicle Speed & Engine RPM at each 10 km
# milestone — one value per channel per instant.
page.add_aggregation(PointValueAggregator(
    name="values_at_distance_milestones",
    input_expressions=[veh_spd, eng_rpm],
    channel_names=["Vehicle Speed", "Engine RPM"],
    event=milestone_event,
    desc="Speed & RPM at each 10 km milestone",
))

# ── POI aggregations ────────────────────────────────────────────────
# Freeze-frame: Engine RPM & Vehicle Speed at each P0301 misfire instant.
# The POI event supplies the timestamps; the sample channels supply the
# values valid there — both series types in one aggregation.
page.add_aggregation(PointValueAggregator(
    name="values_at_p0301",
    input_expressions=[eng_rpm, veh_spd],
    channel_names=["Engine RPM", "Vehicle Speed"],
    event=p0301_event,
    desc="RPM & Speed at each P0301 misfire",
))

print(f"{len(page.aggregations)} aggregations added")

In [0]:
from impulse_reporting.channels.calculated_channel import CalculatedChannel

avg_temp_channel = CalculatedChannel(
    name="avg_temp",
    expr=avg_temp,
    identity={"channel_name": "avg_temp", "data_key": "CALC"},
    desc="Average of ambient and intake air temperature",
)
report.add_calculated_channel(avg_temp_channel)
print("Calculated channel 'avg_temp' registered")

### Numeric POI: fault counts per recording

A numeric POI channel reduces like any signal — but the reductions are
**unweighted** (points have no duration). `count()` / `max()` on `dtc_count`
answer "how many faults did each recording log?" directly from the query engine.

(The report-level `StatsAggregator` is designed for continuous `SampleSeries`
inputs, so a per-container POI count is shown here as a direct query instead.)

In [ ]:
dtc_summary = db.query.select(
    dtc_count.count().alias("n_faults"),
    dtc_count.max().alias("peak_count"),
).solve(spark=spark, solver=report.get_solver())

display(dtc_summary.orderBy("container_id"))

# 7. Compute & Persist

- `determine_report()` — parallel execution
- `persist_results()` — writes star schema

In [0]:
report.determine_report()
report.persist_results()
print(f"Report persisted to {pfx}_*")

# 8. Visualize the Results

Read the Gold-layer tables back and render the
results inline with **matplotlib**:

- **Bar** — RPM histogram
- **Heatmap** — RPM vs Speed
- **Table** — per-container statistics
- **Scatter** — Speed & RPM at each 10 km milestone
  (markers only — values exist only *at* each instant)

Includes two **POI** views: Engine RPM sampled at each P0301 misfire (freeze-frame), and fault-code counts per recording.

In [0]:
import matplotlib.pyplot as plt

# ─── Table prefix ───
T = f"{pfx}"

# ════════════════════════════════════════════════════════════════════
# 1. BAR — RPM Histogram (aggregated across all containers)
# ════════════════════════════════════════════════════════════════════
hist_df = (
    spark.read.table(f"{T}_histogram_fact")
    .join(
        spark.read.table(f"{T}_histogram_dimension").filter("name = 'rpm_histogram'"),
        on="visual_id",
    )
    .groupBy("bin_id", "lower_bound", "upper_bound", "bin_name")
    .agg(F.sum("hist_value").alias("total_duration_us"))
    .orderBy("bin_id")
    .toPandas()
)
hist_df["duration_s"] = hist_df["total_duration_us"] / 1e6

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(hist_df["bin_name"], hist_df["duration_s"], color="steelblue", edgecolor="white")
ax.set_xlabel("Engine RPM bin")
ax.set_ylabel("Duration (s)")
ax.set_title("RPM Histogram — Duration in Each RPM Band (all containers)")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

# ════════════════════════════════════════════════════════════════════
# 2. HEATMAP — RPM vs Speed
# ════════════════════════════════════════════════════════════════════
heat_df = (
    spark.read.table(f"{T}_histogram2d_fact")
    .groupBy("x_bin_id", "y_bin_id", "x_bin_name", "y_bin_name",
             "x_lower_bound", "y_lower_bound")
    .agg(F.sum("hist_value").alias("total_us"))
    .toPandas()
)
heat_df["duration_s"] = heat_df["total_us"] / 1e6

pivot = heat_df.pivot_table(
    index="y_bin_id", columns="x_bin_id",
    values="duration_s", fill_value=0,
)

x_labels = sorted(
    heat_df[["x_bin_id", "x_bin_name"]].drop_duplicates().values,
    key=lambda r: r[0],
)
y_labels = sorted(
    heat_df[["y_bin_id", "y_bin_name"]].drop_duplicates().values,
    key=lambda r: r[0],
)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(
    pivot.values, aspect="auto", origin="lower",
    cmap="YlOrRd", interpolation="nearest",
)
ax.set_xticks(range(len(x_labels)))
ax.set_xticklabels(
    [lbl[1] for lbl in x_labels], rotation=45, ha="right", fontsize=7,
)
ax.set_yticks(range(len(y_labels)))
ax.set_yticklabels([lbl[1] for lbl in y_labels], fontsize=7)
ax.set_xlabel("Engine RPM")
ax.set_ylabel("Vehicle Speed (km/h)")
ax.set_title("RPM vs Speed Heatmap — Duration (s)")
plt.colorbar(im, ax=ax, label="Duration (s)")
plt.tight_layout()
plt.show()

# ════════════════════════════════════════════════════════════════════
# 3. TABLE — Per-container Statistics (container_stats)
# ════════════════════════════════════════════════════════════════════
stats_df = (
    spark.read.table(f"{T}_stats_aggregator_fact")
    .join(
        spark.read.table(f"{T}_stats_aggregator_dimension")
        .filter("name = 'container_stats'"),
        on="visual_id",
    )
    .select("container_id", "channel_name", "aggregation_label", "statistic_value")
    .toPandas()
)

stats_pivot = stats_df.pivot_table(
    index=["container_id", "channel_name"],
    columns="aggregation_label",
    values="statistic_value",
).reset_index()

stats_pivot.columns.name = None
stats_pivot = stats_pivot.sort_values(["container_id", "channel_name"])

print("Per-Container Statistics (full measurement):")
display(
    spark.createDataFrame(
        stats_pivot[["container_id", "channel_name", "min", "median", "mean", "max"]],
    ),
)

# ════════════════════════════════════════════════════════════════════
# 4. SCATTER — Speed & RPM at Each 10 km Milestone
# ════════════════════════════════════════════════════════════════════
milestone_df = (
    spark.read.table(f"{T}_stats_aggregator_fact")
    .join(
        spark.read.table(f"{T}_stats_aggregator_dimension")
        .filter("name = 'values_at_distance_milestones'"),
        on="visual_id",
    )
    .select("container_id", "channel_name", "event_instance_id", "statistic_value")
    .toPandas()
)

milestone_pivot = milestone_df.pivot_table(
    index=["container_id", "event_instance_id"],
    columns="channel_name",
    values="statistic_value",
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
for cid, grp in milestone_pivot.groupby("container_id"):
    ax.scatter(
        grp["Vehicle Speed"], grp["Engine RPM"],
        label=f"Container {cid}", s=60, alpha=0.8, edgecolors="k", linewidths=0.5,
    )
ax.set_xlabel("Vehicle Speed (km/h)")
ax.set_ylabel("Engine RPM")
ax.set_title("Speed & RPM at Each 10 km Milestone")
ax.legend()
plt.tight_layout()
plt.show()

# ════════════════════════════════════════════════════════════════════
# 5. POI — Engine RPM at Each P0301 Misfire (freeze-frame)
# ════════════════════════════════════════════════════════════════════
# PointValueAggregator writes to the shared stats_aggregator_fact table;
# select its visual by name, like the milestone scatter above.
p0301_df = (
    spark.read.table(f"{T}_stats_aggregator_fact")
    .join(
        spark.read.table(f"{T}_stats_aggregator_dimension")
        .filter("name = 'values_at_p0301'"),
        on="visual_id",
    )
    .select("container_id", "channel_name", "event_instance_id", "statistic_value")
    .toPandas()
)

if not p0301_df.empty:
    rpm_hits = p0301_df[p0301_df["channel_name"] == "Engine RPM"]
    fig, ax = plt.subplots(figsize=(10, 4))
    for cid, grp in rpm_hits.groupby("container_id"):
        ax.scatter(
            grp["event_instance_id"], grp["statistic_value"],
            label=f"Container {cid}", s=90, alpha=0.85,
            edgecolors="k", linewidths=0.5,
        )
    ax.set_xlabel("P0301 occurrence #")
    ax.set_ylabel("Engine RPM at fault instant")
    ax.set_title("Freeze-frame: Engine RPM at each P0301 misfire")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No P0301 misfires in the demo data.")


# 10. Cleanup

Drop all tables created by this demo.

In [0]:
if dbutils.widgets.get("drop_created_tables") == "true":
    tables_to_drop = [
        *[f"{pfx}_{t}" for t in SILVER],
        f"{pfx}_histogram_fact",
        f"{pfx}_histogram_dimension",
        f"{pfx}_histogram2d_fact",
        f"{pfx}_histogram2d_dimension",
        f"{pfx}_stats_aggregator_fact",
        f"{pfx}_stats_aggregator_dimension",
        f"{pfx}_event_instance_fact",
        f"{pfx}_event_dimension",
        f"{pfx}_measurement_dimension",
    ]
    for t in tables_to_drop:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
    print(f"Dropped {len(tables_to_drop)} tables")